# ancseq on Google Colab

Run the four cells below from top to bottom to reconstruct ancestral sequences with [ancseq](https://github.com/YuSugihara/ancseq).

- Upload one **aligned** FASTA file containing DNA, amino acid, or codon sequences.
- A CPU runtime is sufficient; no GPU is required.
- The **Run ancseq** cell replaces `/content/ancseq_output` when it is rerun. Download any results you want to keep before the Colab runtime ends.

## Citation

If you use ancseq in your research, please cite:

Sugihara Y, Kourelis J, Contreras MP, Pai H, Harant A, Selvaraj M, Toghani A, Martínez-Anaya C, Kamoun S (2025) *Helper NLR immune protein NRC3 evolved to evade inhibition by a cyst nematode virulence effector*. PLOS Genetics 21(4): e1011653. [https://doi.org/10.1371/journal.pgen.1011653](https://doi.org/10.1371/journal.pgen.1011653)


In [ ]:
# @title 1. Setup
import hashlib
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tarfile
import tempfile
import urllib.request

IQTREE_VERSION = "2.3.6"
IQTREE_URL = (
    "https://github.com/iqtree/iqtree2/releases/download/"
    f"v{IQTREE_VERSION}/iqtree-{IQTREE_VERSION}-Linux-intel.tar.gz"
)
IQTREE_SHA256 = "47389ea3b32c5fb61fba1a2b65c0ea057467d71f39a51fc4c4624927f200786d"
IQTREE_BIN = Path("/usr/local/bin/iqtree")

with tempfile.TemporaryDirectory() as temp_dir:
    temp_path = Path(temp_dir)
    archive_path = temp_path / "iqtree.tar.gz"
    with urllib.request.urlopen(IQTREE_URL, timeout=60) as response:
        with archive_path.open("wb") as archive_file:
            shutil.copyfileobj(response, archive_file)
    actual_sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()
    if actual_sha256 != IQTREE_SHA256:
        raise RuntimeError("The downloaded IQ-TREE archive failed checksum verification.")
    with tarfile.open(archive_path, "r:gz") as archive:
        archive.extractall(temp_path)
    source_bin = (
        temp_path / f"iqtree-{IQTREE_VERSION}-Linux-intel" / "bin" / "iqtree2"
    )
    shutil.copy2(source_bin, IQTREE_BIN)

IQTREE_BIN.chmod(0o755)
pip_environment = os.environ.copy()
pip_environment["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        "git+https://github.com/YuSugihara/ancseq.git@main",
    ],
    check=True,
    env=pip_environment,
)

print("Installed versions:")
subprocess.run(["ancseq", "--version"], check=True)
subprocess.run(["iqtree", "--version"], check=True)


In [ ]:
# @title 2. Configure analysis and upload FASTA
# @markdown Set the options below, run this cell with the **play button (▶)**, and then click **Choose Files** when it appears.
MODE = "DNA"  # @param ["DNA", "AA", "CODON"]
# @markdown ⚠️ **We recommend specifying an outgroup to avoid misinterpretation of ancestral states.**
OUTGROUP = ""  # @param {type:"string", placeholder:"Optional sequence ID"}
USE_FAST = False  # @param {type:"boolean"}
THREADS = 2  # @param {type:"integer", min:1, max:8, step:1}

import re
from pathlib import Path
from Bio import SeqIO
from google.colab import files

if MODE not in {"DNA", "AA", "CODON"}:
    raise ValueError("MODE must be DNA, AA, or CODON.")
if not isinstance(THREADS, int) or THREADS < 1:
    raise ValueError("THREADS must be a positive integer.")

print("Upload one aligned FASTA file using the Choose Files button below.", flush=True)
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Please upload exactly one FASTA file.")

original_name, file_content = next(iter(uploaded.items()))
INPUT_FASTA = Path("/content/ancseq_input.fasta")
INPUT_FASTA.write_bytes(file_content)
OUTPUT_DIR = Path("/content/ancseq_output")

records = list(SeqIO.parse(INPUT_FASTA, "fasta"))
if not records:
    raise ValueError("The uploaded file does not contain any FASTA records.")
sequence_ids = [record.id for record in records]
if len(sequence_ids) != len(set(sequence_ids)):
    raise ValueError("FASTA sequence IDs must be unique.")
alignment_lengths = {len(record.seq) for record in records}
if len(alignment_lengths) != 1:
    raise ValueError("All sequences must have the same aligned length.")
alignment_length = alignment_lengths.pop()
if MODE == "CODON" and alignment_length % 3 != 0:
    raise ValueError("CODON mode requires an alignment length divisible by three.")

OUTGROUP = OUTGROUP.strip()
if OUTGROUP:
    if OUTGROUP not in sequence_ids:
        raise ValueError(f"Outgroup {OUTGROUP!r} is not a FASTA sequence ID.")
    if not re.fullmatch(r"[A-Za-z0-9_.:+-]+", OUTGROUP):
        raise ValueError(
            "The outgroup ID may contain only letters, numbers, _, ., :, +, and -."
        )
else:
    print(
        "⚠️ No outgroup was specified. We recommend specifying an outgroup "
        "to avoid misinterpretation of ancestral states."
    )

print(f"Input: {original_name} ({len(records)} sequences, {alignment_length} sites)")
print(f"Mode: {MODE}; outgroup: {OUTGROUP or 'not specified'}; fast: {USE_FAST}")


In [ ]:
# @title 3. Run ancseq
import shlex
import shutil
import subprocess

if OUTPUT_DIR.exists():
    print(f"Replacing existing output directory: {OUTPUT_DIR}")
    shutil.rmtree(OUTPUT_DIR)

command = [
    "ancseq",
    "--seq",
    str(INPUT_FASTA),
    "--mode",
    MODE,
    "--out",
    str(OUTPUT_DIR),
    "--threads",
    str(THREADS),
]
if USE_FAST:
    command.append("--fast")
if OUTGROUP:
    command.extend(["--outgroup", OUTGROUP])

print("Running:", shlex.join(command), flush=True)
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1,
)
for line in process.stdout:
    print(line, end="", flush=True)
return_code = process.wait()

if return_code != 0:
    print(f"\nancseq failed with exit code {return_code}.")
    print("IQ-TREE diagnostic excerpts (last 80 lines per file):")
    diagnostic_files = [
        ("Tree inference stderr", OUTPUT_DIR / "00_tree" / "00_iqtree.err"),
        ("Tree inference stdout", OUTPUT_DIR / "00_tree" / "00_iqtree.out"),
        ("Ancestral reconstruction stderr", OUTPUT_DIR / "10_asr" / "10_iqtree.err"),
        ("Ancestral reconstruction stdout", OUTPUT_DIR / "10_asr" / "10_iqtree.out"),
        ("INDEL reconstruction stderr", OUTPUT_DIR / "20_indels" / "20_iqtree.err"),
        ("INDEL reconstruction stdout", OUTPUT_DIR / "20_indels" / "20_iqtree.out"),
    ]
    diagnostics_found = False
    for label, diagnostic_path in diagnostic_files:
        if not diagnostic_path.is_file():
            continue
        diagnostic_lines = diagnostic_path.read_text(errors="replace").splitlines()
        if not diagnostic_lines:
            continue
        diagnostics_found = True
        print(f"\n--- {label}: {diagnostic_path} ---")
        print("\n".join(diagnostic_lines[-80:]))
    if not diagnostics_found:
        print("No non-empty IQ-TREE diagnostic files were created.")
    raise RuntimeError(
        "ancseq failed. Read the IQ-TREE diagnostic excerpt above for the cause."
    )

result_dir = OUTPUT_DIR / "30_result"
result_files = sorted(path for path in result_dir.iterdir() if path.is_file())
print("\nResult files:")
for result_file in result_files:
    print(f"- {result_file.name} ({result_file.stat().st_size:,} bytes)")


In [ ]:
# @title 4. Download results
import shutil
from google.colab import files

if not OUTPUT_DIR.is_dir():
    raise FileNotFoundError("Run ancseq before downloading results.")

archive_path = shutil.make_archive(
    "/content/ancseq_results",
    "zip",
    root_dir=OUTPUT_DIR.parent,
    base_dir=OUTPUT_DIR.name,
)
print(f"Created {archive_path}")
files.download(archive_path)
